## Measurement timing, time of day and temperature

**Purpose:** check how the measurements are spaced in time and test whether energy use changes with time of day or outdoor temperature.

**Input:** `fact_energyconsumption.csv` (created by the Dashboard data notebook). This file is not stored in the repository.

**Requires:** pandas, numpy. Random seed: 42 for the shuffled baselines (200 shuffles).

**Summary of findings:** measurements are not at 15-minute intervals; they are single samples at irregular times (median gap between two measurements of the same home: about 30 hours). Costs are therefore reported per measurement, not as monthly or yearly bills. Energy use does not change with time of day (p = 0.56) or outdoor temperature (p = 0.38), and Air Conditioning and Heater do not respond to temperature.

### 1. Build a timestamp

- The `date` and `time` columns are combined into one timestamp.
- Any value that cannot be read becomes empty (`errors="coerce"`), so the count of unread values shows whether the columns are clean.

In [1]:
import pandas as pd
import numpy as np

fact = pd.read_csv("../data/processed/fact_energyconsumption.csv")

# Build a timestamp from the date and time columns
fact["timestamp"] = pd.to_datetime(
    fact["date"].astype(str) + " " + fact["time"].astype(str), errors="coerce"
)
print("Rows:", len(fact), "| unparsed timestamps:", fact["timestamp"].isna().sum())
print(fact["timestamp"].min(), "to", fact["timestamp"].max())

Rows: 100000 | unparsed timestamps: 0
2023-01-01 00:07:00 to 2024-01-01 23:53:00


**Findings**
- 100,000 rows; all timestamps were read (0 unreadable values).
- The data run from 1 January 2023 00:07 to 1 January 2024 23:53 (one year plus one day).

### 2. Are the times on 15-minute marks?

- If the data were recorded every 15 minutes, the minutes would only be 00, 15, 30 or 45.
- If times are random, about 4 of 60 minute values (6.7%) fall on those marks by chance.

In [2]:
minutes = fact["timestamp"].dt.minute
print("Distinct minute values:", minutes.nunique())
print("Share of timestamps on a 15-minute mark (00/15/30/45):",
      round(minutes.isin([0, 15, 30, 45]).mean() * 100, 1), "%")
print("All seconds equal to zero:", bool((fact["timestamp"].dt.second == 0).all()))

Distinct minute values: 60
Share of timestamps on a 15-minute mark (00/15/30/45): 6.6 %
All seconds equal to zero: True


**Findings**
- All 60 minute values occur, and only 6.6% of timestamps fall on a 15-minute mark. This matches random times (6.7%).
- Seconds are always zero, so times are recorded to the minute.
- The measurements are not at 15-minute intervals.

### 3. Gaps between measurements

- For each home, the time between one measurement and the next is calculated (in hours).
- The number of measurements per home per day is counted for days that have at least one measurement. Days with no measurement are not in this count.

In [3]:
fact = fact.sort_values(["home_id", "timestamp"])
gaps_hours = fact.groupby("home_id")["timestamp"].diff().dropna().dt.total_seconds() / 3600
print(gaps_hours.describe().round(2))

per_day = fact.groupby(["home_id", fact["timestamp"].dt.date]).size()
print("Measurements per home per day:")
print(per_day.describe().round(2))

count    99500.00
mean        43.72
std         43.76
min          0.00
25%         12.55
50%         30.22
75%         60.63
max        573.47
Name: timestamp, dtype: float64
Measurements per home per day:
count    77015.00
mean         1.30
std          0.57
min          1.00
25%          1.00
50%          1.00
75%          1.00
max          5.00
dtype: float64


**Findings**
- Gaps between consecutive measurements of the same home: median 30.2 hours (1.3 days), mean 43.7 hours, 25% under 12.6 hours, 75% under 60.6 hours, longest 573 hours (about 24 days).
- 77,015 of the possible 183,000 home-days (500 homes x 366 days, about 42%) have at least one measurement. On those days the median is 1 measurement and the maximum is 5.
- The mean gap (43.7 hours) is consistent with about 200 measurements spread over the 8,784 hours of the period.
- The measurements are separate samples at irregular times, not a continuous series.

**Implication:** the time an appliance runs between measurements is unknown, so no monthly or yearly energy or cost can be calculated. Averages per measurement are used throughout the project.

### 4. Does energy use change with time of day?

- The hour of each measurement (0 to 23) is taken from the timestamp.
- Average kWh per measurement is calculated for each hour.
- To judge whether the differences between hours are real, the spread (highest group mean minus lowest group mean) is compared with the spread obtained when the hour labels are shuffled 200 times (seed 42). If the real spread is not above the shuffled range, the hours differ only by chance.
- The same function is used for temperature in section 5.

In [4]:
fact["hour"] = fact["timestamp"].dt.hour
hourly = fact.groupby("hour")["kwh"].agg(["count", "mean"]).round(3)
print(hourly)

def spread_vs_shuffled(labels, values, n=200, seed=42):
    """Spread (max minus min) of group means: real vs shuffled labels."""
    real = values.groupby(labels).mean()
    real_spread = real.max() - real.min()
    rng = np.random.default_rng(seed)
    sims = []
    for _ in range(n):
        shuf = values.groupby(rng.permutation(labels.values)).mean()
        sims.append(shuf.max() - shuf.min())
    sims = np.array(sims)
    p = (np.sum(sims >= real_spread) + 1) / (n + 1)
    return tuple(float(x) for x in (round(real_spread, 4), round(sims.mean(), 4),
                                    round(np.percentile(sims, 95), 4), round(p, 3)))

print("Hour of day (real spread, shuffled mean, shuffled p95, p):",
      spread_vs_shuffled(fact["hour"], fact["kwh"]))

      count   mean
hour              
0      4225  1.511
1      4030  1.517
2      4229  1.493
3      4099  1.506
4      4284  1.501
5      4201  1.493
6      4253  1.498
7      4224  1.508
8      4123  1.512
9      4083  1.524
10     4179  1.479
11     4226  1.489
12     4201  1.494
13     4143  1.498
14     4186  1.539
15     4094  1.494
16     4122  1.489
17     4088  1.525
18     4229  1.479
19     4090  1.488
20     4069  1.488
21     4243  1.476
22     4138  1.532
23     4241  1.470
Hour of day (real spread, shuffled mean, shuffled p95, p): (0.069, 0.0716, 0.0961, 0.562)


**Findings**
- Each hour has 4,030 to 4,284 measurements. Average kWh per measurement ranges from 1.470 (23:00) to 1.539 (14:00), a spread of 0.069 kWh.
- The shuffled spread averages 0.0716 (95th percentile 0.0961), so the real spread is not above what random labels produce (p = 0.562).
- There is no time-of-day pattern: no peak hours and no quiet hours.

### 5. Does energy use change with outdoor temperature?

- Outdoor temperature is grouped into 5 °C bands, from the lowest temperature up. One extra band is added so that the maximum value (which would otherwise fall outside the last band) is included.
- Average kWh per measurement is calculated for each band and compared with shuffled labels, as in section 4.
- For Air Conditioning and Heater, the correlation between temperature and kWh is calculated. In real homes Air Conditioning would rise in hot weather (positive correlation) and Heater in cold weather (negative correlation).
- The noise range (about +/-0.020 for about 10,000 measurements per appliance) is the size of correlation that chance alone can produce.

In [5]:
lo = int(np.floor(fact["temp"].min() / 5) * 5)
hi = int(np.floor(fact["temp"].max() / 5) * 5) + 10      # one extra band so the maximum is included
fact["temp_band"] = pd.cut(fact["temp"], bins=range(lo, hi, 5), right=False)

print(fact.groupby("temp_band", observed=True)["kwh"].agg(["count", "mean"]).round(3))
print("Rows in a band:", fact["temp_band"].notna().sum(), "of", len(fact))
print("Temperature band (real spread, shuffled mean, shuffled p95, p):",
      spread_vs_shuffled(fact["temp_band"].astype(str), fact["kwh"]))

# Do Air Conditioning and Heater react to temperature?
for a in ["Air Conditioning", "Heater"]:
    s = fact[fact["appliance"] == a]
    r = s["temp"].corr(s["kwh"])
    print(f"{a}: correlation with temperature = {r:.3f} (noise range about +/-{1.96/np.sqrt(len(s)):.3f})")

           count   mean
temp_band              
[-10, -5)   9876  1.516
[-5, 0)    10153  1.485
[0, 5)     10128  1.496
[5, 10)     9992  1.492
[10, 15)    9876  1.493
[15, 20)   10037  1.510
[20, 25)    9887  1.493
[25, 30)    9933  1.501
[30, 35)   10136  1.513
[35, 40)    9889  1.499
[40, 45)      93  1.605
Rows in a band: 100000 of 100000
Temperature band (real spread, shuffled mean, shuffled p95, p): (0.1199, 0.1091, 0.2388, 0.383)
Air Conditioning: correlation with temperature = 0.002 (noise range about +/-0.020)
Heater: correlation with temperature = -0.006 (noise range about +/-0.020)


**Findings**
- Average kWh per measurement is 1.485 to 1.516 across the ten bands from -10 °C to 40 °C (spread 0.031). The last band (40 to 45 °C) has only 93 measurements (mean 1.605), so its average varies a lot by chance.
- With all eleven bands the spread is 0.120, against a shuffled average of 0.109 (95th percentile 0.239), p = 0.383. Without the 93-measurement band, the spread is 0.031 (shuffled average 0.036, p = 0.71). Either way, the real spread is not above what random labels produce.
- Air Conditioning: correlation with temperature 0.002. Heater: -0.006. Both are inside the noise range of +/-0.020.
- There is no temperature effect. The coldest and hottest bands are both slightly higher than the middle, which is chance and not a trend. Air Conditioning does not rise in hot weather and Heater does not rise in cold weather.

## Conclusion and limitations

- Measurements are not at 15-minute intervals. They are separate samples at irregular times: the median gap between two measurements of the same home is about 30 hours. Costs are therefore reported per measurement, not as monthly or yearly bills.
- Energy use does not change with time of day (p = 0.562) or outdoor temperature (p = 0.383), and Air Conditioning and Heater do not respond to temperature (correlations 0.002 and -0.006). Together with the missing seasonal pattern (notebook 01), this supports the view that the dataset is simulated.
- Consequence for the project: time-of-day and temperature views add no information, and time-of-use electricity pricing would give the same average cost as a flat rate in this data. A flat rate is used.
- Limitation: findings apply to this dataset. In real smart home data, time of day, temperature and season would almost certainly matter.